# Gift Recommendation LLM Evaluation

This notebook evaluates different LLM models for gift recommendation generation.

## Metrics Tracked
- **latency_ms**: Time per prompt
- **tokens_in / tokens_out**: Inference cost basis
- **quality_score**: LLM-as-a-judge or rubric scoring
- **monthly_estimate**: Cost for 2B requests

## Models Evaluated
- OpenAI compatible API (e.g., OpenAI, Token Factory)
- Self-hosted vLLM (7B, 13B)

In [1]:
%load_ext autoreload
%autoreload 2


import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "prototype" / "src"))

from src.benchmark_llm import calculate_cost_1m_requests
from src.evaluation import (
    evaluate_quality_with_judge,
    setup_mlflow,
)
from src.generators import generate_gift_recommendation
from src.model_config import (
    calculate_self_hosted_cost,
    get_available_models,
    get_judge_client,
)
from src.prompts import (
    generate_gift_quality_judge_prompt,
)
from src.test_samples import get_test_profiles
from src.utils import load_env_from_repo_root

# Load .env file from repository root
load_env_from_repo_root(".env", override=True)

In [2]:
setup_mlflow("gift_recommendation_eval")

MLflow tracking URI: https://public-tracking-e00-q0ycj5wbge9njs0-p4e20s5hwjds743-mlflow.gw.msp.eu-north1.nebius.cloud
MLflow experiment: <Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1765790976980, experiment_id='1', last_update_time=1765790976980, lifecycle_stage='active', name='gift_recommendation_eval', tags={}>


In [3]:
# Get all available models
models_to_evaluate = get_available_models(
    # model_names=[
    #     "gpt-4o-mini",
    #     "openai-gpt-3.5-turbo",
    #     "tf/gpt-oss-20b",
    #     "tf/DeepSeek-R1-0528",
    # ],
    include_openai=True,
    include_token_factory=True,
    include_self_hosted=True,
)
print(models_to_evaluate)

[<src.model_config.ModelConfig object at 0x11b634590>, <src.model_config.ModelConfig object at 0x11b8f9e80>, <src.model_config.ModelConfig object at 0x11b8f9c70>, <src.model_config.ModelConfig object at 0x11b8f98b0>, <src.model_config.ModelConfig object at 0x11b8f8f80>]


In [4]:
print("Smoke-testing model availability...")

for m in models_to_evaluate:
    try:
        text, metrics = m.client.generate(
            prompt="ping",
            temperature=0.0,
            max_tokens=8,
        )
        print(
            f"✅ {m.name}: reachable (latency_ms={metrics.get('latency_ms'):.1f}, tokens_out={metrics.get('tokens_out')})"
        )
    except Exception as e:
        print(f"❌ {m.name}: {e}")

Smoke-testing model availability...
✅ gpt-4o-mini: reachable (latency_ms=618.3, tokens_out=8)
✅ openai-gpt-3.5-turbo: reachable (latency_ms=500.9, tokens_out=3)
✅ tf/gpt-oss-20b: reachable (latency_ms=619.8, tokens_out=8)
✅ tf/DeepSeek-R1-0528: reachable (latency_ms=654.4, tokens_out=8)
✅ santa-deepseek-r1: reachable (latency_ms=149.0, tokens_out=8)


### Judge Client - gpt-5.2

In [5]:
judge_client = get_judge_client()

In [6]:
# Test judge client with a simple example
kid_profile = get_test_profiles()[0]
print(f"Testing judge with kid profile: {kid_profile.id}")

# Create a sample gift recommendation response
sample_response = """GIFTS:
- LEGO Star Wars set
- Art supplies kit
- Science experiment book

RATIONALE:
Based on the child's interests in building, creativity, and learning, these gifts align well with their profile."""

# Generate judge prompt and evaluate
judge_prompt = generate_gift_quality_judge_prompt(kid_profile, sample_response)
quality_score, quality_rationale = evaluate_quality_with_judge(judge_client, judge_prompt)

print(f"Quality Score: {quality_score}")
print(f"Quality Rationale: {quality_rationale}")

Testing judge with kid profile: aa0b4203-631d-4775-a678-f2cad095e28d
Quality Score: 0.85
Quality Rationale: The recommendation scores high in alignment to the child's interests as it includes items related to building, creativity, and learning, which are reflected in the wishlist. The gifts show creativity and originality by not just repeating the wishlist items but extending them with a LEGO Star Wars set and a science experiment book. The gifts are age-appropriate and safe for a 7-year-old. The output is correct with 3 non-repetitive items and a coherent rationale that ties back to the child's interests. The recommendation is personalized to Emma's profile, making it a high-quality suggestion.


## Evaluate Gift Recommendations

In [7]:
import re
from pathlib import Path

import mlflow
import pandas as pd

test_profiles = get_test_profiles()

results = []
agg_results = []

MLFLOW_AVAILABLE = bool(mlflow.get_tracking_uri())
# Enable automatic tracing for all OpenAI API calls.
# mlflow.openai.autolog()
mlflow.autolog()

# Evaluate all configured models
for model_config in models_to_evaluate:
    model_name = model_config.name
    client = model_config.client
    print(f"Evaluating model: {model_name}")

    with mlflow.start_run(run_name=f"{model_name}"):
        parent_run_id = mlflow.active_run().info.run_id if MLFLOW_AVAILABLE else None
        per_calls = []

        for kid_profile in test_profiles:
            try:
                gift_recommendation, metrics = generate_gift_recommendation(
                    client, kid_profile, temperature=0.7, max_tokens=300
                )

                # TODO: revise/remove
                # Format gift_recommendation as string for judge evaluation
                response_text = "GIFTS:\n" + "\n".join(
                    f"- {gift}" for gift in gift_recommendation.gifts
                )
                response_text += f"\n\nRATIONALE:\n{gift_recommendation.rationale}"

                quality_score = None
                quality_rationale = None
                if judge_client:
                    judge_prompt = generate_gift_quality_judge_prompt(kid_profile, response_text)
                    quality_score, quality_rationale = evaluate_quality_with_judge(
                        judge_client, judge_prompt
                    )

                tokens_in = metrics.get("tokens_in", 0) or 0
                tokens_out = metrics.get("tokens_out", 0) or 0
                latency_ms = metrics.get("latency_ms", 0) or 0

                # Calculate costs: dynamic for self-hosted, fixed for provider models
                if model_config.is_self_hosted and model_config.cost_infra_per_hour:
                    # For self-hosted: calculate cost per 1M tokens based on infrastructure cost
                    cost_per_1m_in, cost_per_1m_out = calculate_self_hosted_cost(
                        latency_ms=latency_ms,
                        tokens_in=tokens_in,
                        tokens_out=tokens_out,
                        cost_infra_per_hour=model_config.cost_infra_per_hour,
                    )
                else:
                    # Provider models: use pre-defined costs per 1M tokens
                    cost_per_1m_in = model_config.cost_per_1m_tokens_in
                    cost_per_1m_out = model_config.cost_per_1m_tokens_out

                # Store cost per 1M tokens (for comparison across models)
                cost_in_1m = round(cost_per_1m_in, 1)
                cost_out_1m = round(cost_per_1m_out, 1)

                # Calculate cost for 1M similar requests
                cost_1m_requests = calculate_cost_1m_requests(
                    cost_per_1m_in=cost_per_1m_in,
                    cost_per_1m_out=cost_per_1m_out,
                    tokens_in=tokens_in,
                    tokens_out=tokens_out,
                )

                record = {
                    "model": model_name,
                    "kid_id": kid_profile.id,
                    "latency_ms": round(metrics.get("latency_ms", 0), 0),
                    "tokens_in": tokens_in,
                    "tokens_out": tokens_out,
                    "quality_score": quality_score,
                    "cost_in_1m": cost_in_1m,
                    "cost_out_1m": cost_out_1m,
                    "cost_1m_requests": cost_1m_requests,
                }
                per_calls.append(record)
                results.append(record)

                if MLFLOW_AVAILABLE:
                    with mlflow.start_run(run_name=f"{kid_profile.id}", nested=True):
                        mlflow.log_metric("latency_ms", record["latency_ms"])
                        mlflow.log_metric("tokens_in", tokens_in)
                        mlflow.log_metric("tokens_out", tokens_out)
                        if quality_score is not None:
                            mlflow.log_metric("quality_score", quality_score)
                        mlflow.log_metric("cost_in_1m", cost_in_1m)
                        mlflow.log_metric("cost_out_1m", cost_out_1m)
                        mlflow.log_metric("cost_1m_requests", cost_1m_requests)
                        mlflow.set_tags(
                            {
                                "kid_id": kid_profile.id,
                                "task": "gift_recommendation",
                            }
                        )
                        mlflow.log_dict(
                            {
                                "gifts": gift_recommendation.gifts,
                                "rationale": gift_recommendation.rationale,
                                "quality_score": quality_score,
                                "quality_rationale": quality_rationale,
                            },
                            "gift_recommendation.json",
                        )
            except Exception as e:
                err_rec = {
                    "model": model_name,
                    "kid_id": kid_profile.id,
                    "error": str(e),
                }
                per_calls.append(err_rec)
                results.append(err_rec)

        df_model = pd.DataFrame([r for r in per_calls if "error" not in r])
        if not df_model.empty:
            agg = {
                "model": model_name,
                "latency_ms": round(df_model["latency_ms"].mean(), 0),
                "tokens_in": df_model["tokens_in"].mean(),
                "tokens_out": df_model["tokens_out"].mean(),
                "quality_score": df_model["quality_score"].mean(),
                "cost_in_1m": round(df_model["cost_in_1m"].mean(), 1),
                "cost_out_1m": round(df_model["cost_out_1m"].mean(), 1),
                "cost_1m_requests": round(df_model["cost_1m_requests"].mean(), 1),
                "calls": len(df_model),
            }
            agg_results.append(agg)

            out_dir = Path("data/evaluation")
            out_dir.mkdir(parents=True, exist_ok=True)
            model_name_path = re.sub(r"[^\w\-]", "-", model_name)
            out_path = out_dir / f"01_gift_rec_eval-{model_name_path}.csv"
            df_model.to_csv(out_path, index=False)
            print(f"Saved per-call results for {model_name} -> {out_path}")

            if MLFLOW_AVAILABLE:
                mlflow.log_metric("latency_ms", agg["latency_ms"])
                mlflow.log_metric("tokens_in", agg["tokens_in"])
                mlflow.log_metric("tokens_out", agg["tokens_out"])
                mlflow.log_metric("quality_score", agg["quality_score"])
                mlflow.log_metric("cost_in_1m", agg["cost_in_1m"])
                mlflow.log_metric("cost_out_1m", agg["cost_out_1m"])
                mlflow.log_metric("cost_1m_requests", agg["cost_1m_requests"])
                mlflow.log_metric("calls", agg["calls"])
                mlflow.set_tags({"task": "gift_recommendation"})
        else:
            print(f"No successful calls for model {model_name}")

2025/12/16 18:52:02 INFO mlflow.tracking.fluent: Autologging successfully enabled for openai.


Evaluating model: gpt-4o-mini
🏃 View run 8a3463b7-fe8f-43f2-a3ec-990cbe4f03b3 at: https://public-tracking-e00-q0ycj5wbge9njs0-p4e20s5hwjds743-mlflow.gw.msp.eu-north1.nebius.cloud/#/experiments/1/runs/3e1e033866c44b7e97a2bee5a8037019
🧪 View experiment at: https://public-tracking-e00-q0ycj5wbge9njs0-p4e20s5hwjds743-mlflow.gw.msp.eu-north1.nebius.cloud/#/experiments/1
🏃 View run 99ddb802-821c-4dc7-9531-2ee4a721aa5b at: https://public-tracking-e00-q0ycj5wbge9njs0-p4e20s5hwjds743-mlflow.gw.msp.eu-north1.nebius.cloud/#/experiments/1/runs/8c4fc6ccadfe419ea717f85d35e10343
🧪 View experiment at: https://public-tracking-e00-q0ycj5wbge9njs0-p4e20s5hwjds743-mlflow.gw.msp.eu-north1.nebius.cloud/#/experiments/1
🏃 View run 6607d5c2-8872-4f48-adce-2e6d4a600793 at: https://public-tracking-e00-q0ycj5wbge9njs0-p4e20s5hwjds743-mlflow.gw.msp.eu-north1.nebius.cloud/#/experiments/1/runs/dedb271c0902423f95377d7e505b5ead
🧪 View experiment at: https://public-tracking-e00-q0ycj5wbge9njs0-p4e20s5hwjds743-mlflow.gw

In [8]:
# Final aggregated summary
df_results = pd.DataFrame(results)
df_agg = pd.DataFrame(agg_results)
print("Per-model aggregate summary:")
df_agg.round(3)

Per-model aggregate summary:


,model,latency_ms,tokens_in,tokens_out,quality_score,cost_in_1m,cost_out_1m,cost_1m_requests,calls
0,gpt-4o-mini,3076.0,220.00,131.50,0.85,0.1,0.6,111.9,4
1,openai-gpt-3.5-turbo,918.0,224.00,90.00,0.85,1.5,2.0,516.0,4
2,tf/gpt-oss-20b,1236.0,282.00,267.25,0.85,0.1,0.6,202.6,4
3,tf/DeepSeek-R1-0528,8526.0,226.25,101.00,0.85,0.8,2.4,423.4,4
4,santa-deepseek-r1,662.0,222.00,76.25,0.85,2.4,7.2,1095.6,4


In [9]:
cost_per_1m_in, cost_per_1m_out

(2.1594930995811334, 7.37549950933864)